# Tinh chỉnh bộ đọc chữ (OCR) tiếng Việt cho app Bé Nốc

Notebook này **tự chạy từ đầu đến cuối trên Google Colab (GPU T4 miễn phí)**, không cần tải thêm gì.

**Vấn đề cần giải quyết:** mô hình gốc `latin_PP-OCRv5_mobile_rec` chỉ có lớp đầu ra cho 44/134 chữ tiếng Việt có dấu, nên các chữ như `ạ ả ế ề ể ệ ọ ố ớ ụ ứ ự ỳ ỷ` bị **bỏ mất** khi đọc ảnh chat.

**Cách làm:**
1. Lấy các câu văn bản pháp luật có dấu (1.301 câu, còn ≈930 sau khi lọc nhãn rác) từ dataset **niits/vietnamese-legal-ocr** (đã nhúng sẵn trong notebook; dataset gốc chỉ có nhãn chữ, không có ảnh) + câu chat đặt món + tên món Bé Nốc + từ hiếm (để đủ cả 10 chữ không hề có trong dataset pháp luật).
2. **Vẽ ảnh** từng dòng chữ giống bong bóng Messenger/Zalo (nhiều font, nền sáng/tối, mờ, nhiễu, nén JPEG).
3. **Nới từ điển** từ 836 lên 926 ký tự: giữ nguyên trọng số cũ, chữ mới khởi tạo từ chữ không dấu tương ứng.
4. Tinh chỉnh (fine-tune) mô hình, xuất **ONNX**, chấm điểm bằng CER.
5. Tải về `rec.onnx` + `rec_dict.txt` + `model.txt` để thay vào app (không dùng Gemini cho phần đọc chữ).

**Thời gian:** mặc định 120.000 dòng × 20 epoch ≈ 1,5–2 giờ trên T4. Muốn thử nhanh: đặt `N_TRAIN=20000`, `EPOCHS=3` ở cell 1.

> Lưu ý trung thực cho đồ án: ảnh train/eval ở đây là **ảnh mô phỏng**. Số liệu CER là cận trên tốt hơn thực tế; kết quả thật phải đo trên ảnh chat thật của quán (bảng `ocr_mau` ở Supabase Test).

## 0. Cài đặt
Bật GPU: **Runtime → Change runtime type → T4 GPU**, rồi chạy cell dưới.

In [ ]:
# === 0. Cài đặt (≈ 3–5 phút). Chọn Runtime → Change runtime type → GPU (T4) trước khi chạy ===
import subprocess, sys, os
def sh(cmd, quiet=False):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if not quiet: print(line, end='')
    if p.wait() != 0: raise RuntimeError(f'Lệnh lỗi ({p.returncode}): {cmd}')

sh('nvidia-smi -L || echo "KHÔNG có GPU — hãy bật GPU trong Runtime → Change runtime type"')
sh('apt-get -qq update && apt-get -qq install -y fonts-roboto fonts-open-sans fonts-noto-core fonts-dejavu-core fonts-liberation', quiet=True)
sh('pip -q install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
if not os.path.exists('/content/PaddleOCR'):
    sh('cd /content && git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git')
sh('cd /content/PaddleOCR && pip -q install -r requirements.txt paddle2onnx onnxruntime fonttools pyyaml onnx_graphsurgeon --extra-index-url https://pypi.ngc.nvidia.com')
os.chdir('/content/PaddleOCR')
import paddle; paddle.utils.run_check()


## 1–2. Dữ liệu
Đọc dataset nhúng sẵn, ghép với câu chat đặt món và từ hiếm.

In [ ]:
# Dữ liệu nhúng sẵn (nén gzip+base64) — không cần tải gì thêm
LEGAL_B64 = '...'   # niits/vietnamese-legal-ocr: 1301 câu khác nhau
DICT_B64  = 'H4sIAPu8+2oC/62WRXIDSxZF52cXzczMzMzMzMwQsqySUjLKTJKZmdmOqERNylvo0Ab+X0Jfzbtnf3CiQKq87777qiJfy+t4PW/gjbyJN/MW3srbeB/v5wN8kA/xYT7CR/kYH+cTfJJP8Wk+w2f5HJ/nC3yRL/FlvsJX+Rpf53t8nx/wQ37Ej/kJP+Vn/Jxf8Et+xa/5Db/ld/yeP/BH/sSf+Qt/5W/8HVvC9mHL2H5sBZthq9ga1mDr2AZ2ADuIHcIOY0ewo9gmdgw7jp3ATmKnsNPYGewsdg47j13AtrBt7CJ2CbuMXcGuYtew69gN7CZ2C7uN3cHuYvew+9gD7CH2CHuMPcGeYs+w59gL7CX2CnuNvcHeYu+w99gH7COuhOvDlXH9uAouw1VxNZzB1XEN3ABuEDeEG8aN4EZxTdwYbhw3gZvETeGmcTO4Wdwcbh63gGvh2rhF3BJuGbeCW8Wt4dZxG7hN3BZuG7eD28Xt4fZxB7hD3BHuGHeCO8Wd4c5xF7hL3BXuGneDu8Xd4e5xD7hHfAnfhy/j+/EVfIav4mt4g6/jG/gB/CB+CD+MH8GP4pv4Mfw4fgI/iZ/CT+Nn8LP4Ofw8fgHfwrfxi/gl/DJ+Bb+KX8Ov4zfwm/gt/DZ+B7+L38Pv4w/wh/gj/DH+BH+KP8Of4y/wl/gr/DX+Bn+Lv8Pf4x/wj4QSoY9QJvQTKoSMUCXUCIZQJzQIA4RBwhBhmDBCGCU0CWOEccIEYZIwRZgmzBBmCXOEecICoUVoExYJS4RlwgphlbBGWCdsEDYJW4Rtwg5hl7BH2CccEA4JR4RjwgnhlHBGOCdcEC4JV4Rrwg3hlnBHuCc8EB6JJWIfsUzsJ1aIGbFKrBENsU5sEAeIg8Qh4jBxhDhKbBLHiOPECeIkcYo4TZwhzhLniPPEBWKL2CYuEpeIy8QV4ipxjbhO3CBuEreI28Qd4i5xj7hPPCAeEo+Ix8QT4inxjHhOvCBeEq+I18Qb4i3xjnhPfCA+kkqkPlKZ1E+qkDJSlVQjGVKd1CANkAZJQ6Rh0ghplNQkjZHGSROkSdIUaZo0Q5olzZHmSQukFqlNWiQtkZZJK6RV0hppnbRB2iRtkbZJO6Rd0h5pn3RAOiQdkY5JJ6RT0hnpnHRBuiRdka5JN6Rb0h3pnvRAeqRTotNHp0ynn06FTkanSqdGx9Cp02nQGaAzSGeIzjCdEZ7H83kBL+RFvJiX8FJexst5Ba/kVbya1/B23sE7eRfv5j28l2/wTb7Fd/gu/+Cf/Itv82/ybfIDupW6mBIbone9QtdU6S6MiwkxLWbEJN2ZphijOz1Cd2pH7Or8TscLHUt096rka+Qb5JvkW+Q75Lvke+SH5EfkJ+Sn5Gfk5+QX5Ffk1+Q35Lfkd+T35A/kj71PaLc0KiRWklhJRZQkXlKRpd5vKqY0K+bEvFgQLdEWi2JJLItVsSZkrLQpjsSZOBcquHQprsS1uBG3QmZK9+JBTZB2Wc+XtVZZ/ynrflnPlx/J1aOybK2Sr9PNJJjpT5nuZhLN9FDWuy/hTMLZltgW6limjmUqJDsWJ+JUaNFMRWUqKlNRmYrKJJipqExFtdXZdp8oi35REZlQSu2aMEIpthtiQAyKITEslFRbzWyrmW0ZaivVlnRa0mlJpyWdlnRa0mnJfEsmWzLfeqTblK+mfDXlqylfTflqyldTvpry1ZSvpnxVtXZVa1e1dlVBVaVXVWg13avpvKZzIw9GtRvVbuTXyK9RaEahGa1h1Bsjf0b9MXtiX2gazaFQv4w0jLwYhW4UulHoRj006qFRD408GXky8mA0KEaDUldf6vJQl4e6aq+r9oY0Gqqj0TuXz4Z8Tul3o2sz0Psad0tasnT6HG0H/ude4DnZCPyfXUDRpBijGKeYoJikmKKYppihmKWYo5inWKBoUbQpFimWKJYpVinWKNYpNig2KbYotimOKU4oTinOKM4pLiguKa4oriluKG4p7ijuKR4oHnkq8dTHUz9PZZ4qPGU8VXmq8WR46n1eFGpF4Q1qCIY0bEPnPNvSVD3b0kfmP+u993OdZ67ueeZaMZRWerucrtEkGb3mAxn5JXkva82X0QtulKnpZaa5rGgOZ274Lzb35o04CgAA'   # từ điển gốc latin_PP-OCRv5
MENU_B64  = 'H4sIAPu8+2oC/4VWzUocQRB+lcKTkkA0YJKzBhTUQKIBIeQwOzvZaWa7Z+LOmDVXDyEEQQk5hBCyK0hY46JCLu4ce9n36DdJVfdMz5+ai05/VV2/X1Xvm7nV2ZCD74Xg+kyeC+jLm7mHYOCOHMA+oRYRHkfFxIE9OSzQ2Ffplxha00P83lPpSHQKGZ6/C1iXAwEvE5WeuFZkvKIx0VHpUVxxW8O35MUtQRJaiZGARogEZlHhxYmLf8aubyW+mowjcNVkxI2hzAkpbDDeciLUmV2p9KfJKsO2WL90Kq68KLvHDEIB80/AlUO20BRbwzD/rNDZ8EOHQddBOJAjUYLU5NKFyJd/gTtMm+vgd1aU1mxI0LYcxY1EdsKgFQaBvrOWVwx62I64Xup1lX5Fz/EeqqHCpUOfp2RMnibaVkmCxRu6PrRV+sepy4pbMD3J85yeeDqgbaMzu5oN2X9U77AZeJZsxj8IuvnDdOmeS67veT2PlDZVeubek2lD3g/lIKs7lssFv1Srd0m3G4dRxEoBuL4jfNhH7neAI8/iUp1LCkgDpH1LTc5iiFFNp71OtTG5jTA3bD0eqN5aGGqbyASBdZKD0Fgq9a+r0usELZP9nCx+CK+S1kH+3SfZ9ARNMhByGNZSbs+uHOjqmrK6iGqJkV7zIhOHZ5EYoTw9sLK2aSxqcJXeWFhgDLEc49hR1Xl2o8k/Q9NAjwDHjG9tzW1KhmSGHHfLMxLeprCK5nHczi0XLaCdErJCEwB9lX5OKgqcdh2rQES+tpqkIawk1Lgted6MIut6D/uBG8Hc1LvKibHFuBXi2KufdYEboCVrhvaKajR0MbQS5obdNnzwWS/Q8TuuExaOzZEzEReLiepVaNSx3LpuekuOiQphNVPiNW0i5jT6dreamXFPmAEzCK4OZohNfKIFQXNDsudEZzRwloAcRfB0cZF3TT5cA8s5YLzTXIxNUT0i9tB6fJ/QoFZH2J52fNziOgAL7TOVforr11T6jdmDHhx74tNDTNhFhhfq9WEsbY92Njivox77qBuwnf1fxf10JGz5kUk4gfJCi0LeMh7NysJNgdL8Muxu6kYiYQseb1R254ZXeeuN8SWkq5r8FrkXg3bMkDafmp1sfaMsQqoQtGsf6NLrtVZ+46sv6GP9bOq90fh5sIlsiaBPRfuVRZkl6+K+fVSuBAT6HS/lD/MPloOFoq54Xlo0gC6LnpT8VN0a0+PSz4XpMVLn7T+D43SDaAkAAA=='   # tên món Bé Nốc


In [ ]:
# === 1. Tham số + dữ liệu gốc (đã nhúng sẵn trong notebook) ===
import os, re, json, gzip, base64, random, glob, unicodedata, collections, math, subprocess, sys
WORK     = os.environ.get('OCR_WORK', '/content/ocr_vi')      # thư mục làm việc
N_TRAIN  = int(os.environ.get('N_TRAIN', 120000))              # số dòng ảnh để train
N_EVAL   = int(os.environ.get('N_EVAL', 4000))                 # số dòng ảnh để chấm
EPOCHS   = int(os.environ.get('EPOCHS', 20))                   # T4 ≈ 4–6 phút/epoch với 120k dòng
BATCH    = int(os.environ.get('BATCH', 128))
LR       = float(os.environ.get('LR', 0.0003))
os.makedirs(WORK, exist_ok=True)

def unb(b): return gzip.decompress(base64.b64decode(b)).decode('utf-8')
LEGAL = [s for s in unb(LEGAL_B64).split('\n') if s]            # niits/vietnamese-legal-ocr (câu văn bản pháp luật)
DICT0 = unb(DICT_B64).split('\n')                               # từ điển gốc latin_PP-OCRv5 (836 ký tự)
if DICT0[-1] == '': DICT0.pop()
MENU  = json.loads(unb(MENU_B64))                               # tên món của Bé Nốc
print(f'câu luật: {len(LEGAL)} · từ điển gốc: {len(DICT0)} ký tự · món: {len(MENU)}')

VN = "àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ"
VN_ALL = sorted(set(VN + VN.upper()))                           # 134 chữ cái có dấu / đ
MISSING = [c for c in VN_ALL if c not in set(DICT0)]
print(f'chữ tiếng Việt chưa có trong từ điển gốc: {len(MISSING)}/134 →', ''.join(MISSING))


In [ ]:
# === 2. Tạo kho câu: văn bản pháp luật (dataset) + câu chat đặt món + từ hiếm ===
rnd = random.Random(7)
DICT_VI = DICT0 + MISSING                                      # chữ mới nối vào CUỐI (giữ nguyên chỉ số lớp cũ), dấu cách sẽ tự thêm sau cùng
CHARSET = set(DICT_VI) | {' '}

def chunk(s, lo=10, hi=34):
    out, cur, tgt = [], '', rnd.randint(lo, hi)
    for w in s.split():
        t = (cur + ' ' + w).strip()
        if len(t) <= tgt or not cur: cur = t
        else: out.append(cur); cur, tgt = w, rnd.randint(lo, hi)
    if cur: out.append(cur)
    return out

VOW = set('aeiouyàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ')
def sach(w):                                                    # bỏ từ rác do OCR nhãn sai (không có nguyên âm, lẫn ký hiệu lạ)
    a = [c for c in w.lower() if c.isalpha()]
    return len(a) < 4 or any(c in VOW for c in a)
LEGAL = [s for s in LEGAL if all(sach(w) for w in s.split())]
print('câu luật sau khi lọc nhãn rác:', len(LEGAL))
legal_lines = [c for s in LEGAL for c in chunk(s)]

MON = [m for m in MENU if not re.search(r'upsize|size|chọn|topping|combo|thêm|^đường$|^đá$', m, re.I)]
MON = list(dict.fromkeys(MON))
SIZE = ['M', 'L']; DUONG = ['100%', '70%', '50%', '30%', 'ít ngọt', 'ít đường', 'không đường']
DA = ['ít đá', 'nhiều đá', 'không đá', 'đá riêng']
TOPPING = ['thêm trân châu', 'thêm thạch dừa', 'thêm kem cheese', 'thêm sương sáo', 'thêm hạt chia', 'thêm kem trứng']
SO = ['1', '2', '3', '4', '5', 'một', 'hai', 'ba', 'bốn']
PHO = ['Quang Trung', 'Nguyễn Trãi', 'Lê Văn Lương', 'Trần Phú', 'Hoàng Quốc Việt', 'Phạm Văn Đồng', 'Đại Cồ Việt',
       'Nguyễn Văn Cừ', 'Thái Hà', 'Láng Hạ', 'Tôn Thất Tùng', 'Đội Cấn', 'Kim Mã', 'Hồ Tùng Mậu', 'Phạm Hùng', 'Cầu Giấy']
TEN = ['Hương', 'Thảo', 'Ngọc Ánh', 'Minh Đức', 'Phương Anh', 'Quốc Bảo', 'Thuỳ Dương', 'Hoàng Nam', 'Trà My', 'Khánh Linh',
       'Việt Thái', 'Bích Ngọc', 'Tuấn Anh', 'Đặng Huệ', 'Lê Vỹ', 'Mỹ Duyên', 'Thanh Tùng', 'Ngô Ý', 'Hạnh', 'Vũ Quỳnh']
KH = ['Em ơi cho chị {q} {m}', 'Cho mình {q} {m} {da} nha', '{q} {m} size {s} {da} giúp chị', 'Lấy {q} {m}, {duong} nhé',
      '{m} {da}, {duong} nha em', 'Thêm {q} {m} nữa nhé', 'Cho mình {q} {m} {tp}', 'Chị lấy {q} {m} và {q2} {m2}',
      'Ship về {n} ngõ {n2} {pho} giúp chị', 'Mình qua lấy lúc {h}h{p} được không ạ', 'Chị lấy lúc {h}h{p} chiều nhé',
      'Đơn này ghi tên {ten} giúp chị', 'sđt 09{d1} {d2} {d3} nha', 'ok e, ck luôn nhé', 'Có còn {m} không em?',
      'Huỷ giúp chị ly {m} nhé', 'Ủa sao chưa thấy ship vậy em', 'Bé Nốc ơi hôm nay mở cửa mấy giờ?', 'Địa chỉ: số {n} {pho}, Hà Nội',
      'Mình ở {pho}, phí ship bao nhiêu vậy ạ?', 'Cho {q} {m} mang về nhé', 'Đổi {m} sang size {s} được không em']
QU = ['Dạ quán nhận đơn ạ', 'Dạ còn ạ, mình lấy mấy phần ạ?', 'Tổng của mình là {gia}đ ạ', 'Quán giao lúc {h}h{p} nha',
      'Dạ em ghi chú rồi ạ', 'Mình chuyển khoản giúp em nhé', 'Dạ hôm nay quán mở 7h đến 22h ạ', 'Em gửi bếp rồi nha chị',
      'Dạ {m} hết rồi ạ, mình đổi món khác được không ạ?', 'Dạ phí ship {phi}đ ạ', 'Đơn của mình {q} {m}, tổng {gia}đ ạ',
      'Cảm ơn chị đã ủng hộ quán ạ', 'Dạ em xác nhận đơn cho {ten} nhé']

def fill(t):
    d = dict(q=rnd.choice(SO), q2=rnd.choice(SO), m=rnd.choice(MON), m2=rnd.choice(MON), s=rnd.choice(SIZE), da=rnd.choice(DA),
             duong=rnd.choice(DUONG), tp=rnd.choice(TOPPING), n=rnd.randint(1, 200), n2=rnd.randint(1, 60), pho=rnd.choice(PHO),
             h=rnd.randint(7, 21), p=rnd.choice(['', '15', '30', '45', '05']), ten=rnd.choice(TEN),
             d1=rnd.randint(10, 99), d2=rnd.randint(100, 999), d3=rnd.randint(100, 999),
             gia=f'{rnd.randint(2, 30) * 5}.000', phi=f'{rnd.choice([5, 10, 15, 20])}.000')
    return t.format(**d)

chat_lines = [fill(rnd.choice(KH + QU)) for _ in range(30000)]
chat_lines += [m for m in MON for _ in range(4)]               # tên món đứng riêng (thực đơn)
chat_lines += [f'{rnd.choice(["Hôm nay", "Hôm qua", "T5", "T6", "CN"])} lúc {rnd.randint(6, 23)}:{rnd.randint(0, 59):02d}' for _ in range(500)]

# Từ hiếm: ép mô hình thấy đủ mọi chữ có dấu (kể cả 10 chữ không hề có trong dataset pháp luật)
RARE = ('õ lõm mõ sõi dõi lõi nõn bõ ũ mũi cũ ngũ hũ lũ sũng tũn Ằ NHẰM BẰNG HẰNG VẰNG CẰN ẵ ẵm bồng_ẵm Ẵ ẴM '
        'Ẹ ẹ mẹ đẹp nẹp ẸP ĐẸP MẸ chẹp Ẽ ẽ BẼ LẼ CHẼ KHẼ Ỵ ỵ kỵ tỵ KỴ TỴ kỵ_sĩ '
        'ặ ẳ ẫ ẩ ỡ ợ ữ ự ỹ ỷ ễ ể ệ ỗ ổ ộ ẫ ậ Ặ Ẳ Ẫ Ẩ Ỡ Ợ Ữ Ự Ỹ Ỷ Ễ Ể Ệ Ỗ Ổ Ộ').split()
RARE = [w.replace('_', ' ') for w in RARE]

words = collections.Counter(w for l in legal_lines + chat_lines for w in l.split())
cf = collections.Counter(c for l in legal_lines + chat_lines for c in l)
def wt(w):
    r = [1 / max(cf.get(c, 0), 1) for c in w if c not in 'abcdefghijklmnopqrstuvwxyz0123456789 ']
    return (max(r) if r else 0.0005) ** 0.5
vocab = list(words); weights = [wt(w) for w in vocab]
def rand_words():
    n = rnd.randint(2, 6)
    ws = rnd.choices(vocab, weights, k=n)
    if rnd.random() < 0.5: ws[rnd.randrange(n)] = rnd.choice(RARE)
    s = ' '.join(ws)
    return s.upper() if rnd.random() < 0.1 else s

def build(n):
    out = []
    for _ in range(n):
        r = rnd.random()
        cat, s = ('chat', rnd.choice(chat_lines)) if r < 0.40 else ('luat', rnd.choice(legal_lines)) if r < 0.75 else ('tu_ngau_nhien', rand_words())
        s = ' '.join(s.split())
        if rnd.random() < 0.05: s = s.upper()
        if rnd.random() < 0.03: s = s.lower()
        if 2 <= len(s) <= 60 and set(s) <= CHARSET: out.append((cat, s))
    return out

texts_train, texts_eval = build(N_TRAIN), build(N_EVAL)
cc = collections.Counter(c for _, s in texts_train for c in s)
print(f'train {len(texts_train)} dòng · eval {len(texts_eval)} dòng')
print('chữ có dấu ít gặp nhất trong tập train:', sorted(((cc[c], c) for c in VN_ALL))[:8])
print('mẫu:', [s for _, s in texts_train[:5]])


## 3. Vẽ ảnh
Tạo ảnh từng dòng chữ + file nhãn `train_list.txt` / `eval_list.txt` (khoảng 3–6 phút).

In [ ]:
# === 3. Vẽ ảnh từng dòng chữ giống bong bóng chat (nhiều font, nền sáng/tối, mờ, nhiễu, nén JPEG) ===
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from multiprocessing import Pool
import numpy as np
from fontTools.ttLib import TTFont

def find_fonts():
    cands = []
    for pat in ['/usr/share/fonts/**/*.ttf', '/usr/local/share/fonts/**/*.ttf', os.path.expanduser('~/.fonts/*.ttf'),
                '/content/fonts/*.ttf', os.environ.get('FONT_DIR', '/nonexistent') + '/*.ttf']:
        cands += glob.glob(pat, recursive=True)
    ok = []
    for f in sorted(set(cands)):
        try:
            cmap = TTFont(f, fontNumber=0, lazy=True).getBestCmap()
            if all(ord(c) in cmap for c in VN_ALL + list('0123456789 .,:;!?-+%()/@#đĐ')): ok.append(f)
        except Exception: pass
    return ok
FONTS = find_fonts()
print(f'{len(FONTS)} font đủ chữ tiếng Việt:', [os.path.basename(f) for f in FONTS][:12])
assert FONTS, 'Chưa có font tiếng Việt: chạy cell cài đặt (apt fonts-*) trước'

THEMES = [  # (nền, chữ): Messenger/Zalo sáng - tối - bong bóng xanh
    ((255, 255, 255), (5, 5, 5)), ((240, 240, 240), (5, 5, 5)), ((10, 124, 255), (255, 255, 255)), ((0, 132, 255), (255, 255, 255)),
    ((48, 48, 48), (228, 230, 235)), ((0, 0, 0), (228, 230, 235)), ((219, 235, 255), (8, 18, 37)), ((226, 233, 241), (8, 18, 37)),
    ((43, 46, 51), (230, 232, 235)), ((22, 64, 125), (230, 232, 235)), ((255, 255, 255), (101, 103, 107)), ((250, 250, 245), (30, 30, 30)),
]

def render(args):
    i, text, path = args
    r = random.Random(i * 7919 + 13)
    bg, fg = r.choice(THEMES)
    bg = tuple(max(0, min(255, c + r.randint(-6, 6))) for c in bg)
    font = ImageFont.truetype(r.choice(FONTS), r.randint(26, 46))
    l, t, rr, b = font.getbbox(text)
    px, py = r.randint(8, 30), r.randint(6, 16)
    im = Image.new('RGB', (rr - l + 2 * px, b - t + 2 * py), bg)
    ImageDraw.Draw(im).text((px - l, py - t), text, font=font, fill=fg)
    if r.random() < 0.5: im = im.rotate(r.uniform(-1.2, 1.2), expand=True, fillcolor=bg, resample=Image.BICUBIC)
    if r.random() < 0.6:                                        # ảnh chụp màn hình bị thu nhỏ rồi phóng lại
        k = r.uniform(0.45, 0.9); w, h = im.size
        im = im.resize((max(8, int(w * k)), max(8, int(h * k))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if r.random() < 0.4: im = im.filter(ImageFilter.GaussianBlur(r.uniform(0.3, 1.0)))
    if r.random() < 0.5:
        a = np.asarray(im).astype(np.int16) + np.random.RandomState(i % 2**31).normal(0, r.uniform(1, 5), (im.size[1], im.size[0], 1))
        im = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    im.save(path, quality=r.randint(60, 95))
    return path

def make_set(name, items):
    d = f'{WORK}/{name}'; os.makedirs(d, exist_ok=True)
    jobs = [(i + (0 if name == 'train' else 10_000_000), s, f'{d}/{i:06d}.jpg') for i, (_, s) in enumerate(items)]
    with Pool(os.cpu_count()) as p: p.map(render, jobs, chunksize=200)
    with open(f'{WORK}/{name}_list.txt', 'w', encoding='utf-8') as f:
        for (cat, s), j in zip(items, jobs): f.write(f'{name}/{os.path.basename(j[2])}\t{s}\n')
    with open(f'{WORK}/{name}_cat.json', 'w') as f: json.dump([c for c, _ in items], f)

make_set('train', texts_train); make_set('eval', texts_eval)
open(f'{WORK}/dict_vi.txt', 'w', encoding='utf-8').write('\n'.join(DICT_VI) + '\n')
print('xong ảnh:', len(texts_train), '+', len(texts_eval))


## 4. Nới từ điển
Thêm lớp đầu ra cho các chữ tiếng Việt còn thiếu mà vẫn giữ kiến thức của mô hình gốc.

In [ ]:
# === 4. Nới lớp đầu ra (CTC) của mô hình gốc để có thêm lớp cho các chữ tiếng Việt còn thiếu ===
# Giữ nguyên toàn bộ phần "mắt" (backbone, neck) và các lớp chữ cũ; lớp chữ mới khởi tạo bằng cột của chữ không dấu tương ứng.
import paddle, urllib.request
PRE_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams'
PRE = f'{WORK}/latin_pre.pdparams'
if not os.path.exists(PRE): urllib.request.urlretrieve(PRE_URL, PRE)
sd = paddle.load(PRE)
W, B = sd['head.ctc_head.fc.weight'].numpy(), sd['head.ctc_head.fc.bias'].numpy()   # W: [120, 838] = blank + 836 chữ + dấu cách
assert W.shape[1] == len(DICT0) + 2, f'từ điển gốc ({len(DICT0)}) không khớp mô hình ({W.shape[1]} lớp)'
nrs = np.random.RandomState(0)
newW, newB = [], []
for ch in MISSING:
    base = 'd' if ch == 'đ' else 'D' if ch == 'Đ' else unicodedata.normalize('NFD', ch)[0]
    j = DICT0.index(base) + 1 if base in DICT0 else 0
    newW.append(W[:, j] + nrs.normal(0, 0.02, W.shape[0])); newB.append(B[j] - 0.5)
W2 = np.concatenate([W[:, :-1], np.stack(newW, 1), W[:, -1:]], 1).astype('float32')    # dấu cách vẫn nằm cuối cùng
B2 = np.concatenate([B[:-1], np.array(newB), B[-1:]]).astype('float32')
sd['head.ctc_head.fc.weight'] = paddle.to_tensor(W2); sd['head.ctc_head.fc.bias'] = paddle.to_tensor(B2)
for k in [k for k in sd if k.startswith('head.gtc_head')]: del sd[k]                  # nhánh NRTR chỉ dùng khi train, học lại từ đầu
paddle.save(sd, f'{WORK}/init_vi.pdparams')
print('lớp CTC:', W.shape[1], '→', W2.shape[1], '| khởi tạo:', f'{WORK}/init_vi.pdparams')


## 5. Cấu hình
`SAVE_TO_DRIVE = True` nếu muốn lưu checkpoint lên Google Drive (mất kết nối vẫn train tiếp được).

In [ ]:
# === 6. Cấu hình train (lấy cấu hình gốc của PaddleOCR rồi chỉnh) ===
import yaml, paddle
USE_GPU = paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', 4 if USE_GPU else 0))
SAVE_TO_DRIVE = False        # True: lưu checkpoint lên Google Drive để mất kết nối vẫn train tiếp được
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/ocr_vi_output'
else: OUT = f'{WORK}/output'
os.makedirs(OUT, exist_ok=True)
print('GPU:', USE_GPU, '| checkpoint lưu ở', OUT)
cfg = yaml.safe_load(open('configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml', encoding='utf-8'))
G = cfg['Global']
G.update(use_gpu=USE_GPU, epoch_num=EPOCHS, save_model_dir=OUT, save_epoch_step=1, eval_batch_step=[0, 1000],
         pretrained_model=f'{WORK}/init_vi.pdparams', checkpoints=None, character_dict_path=f'{WORK}/dict_vi.txt',
         use_space_char=True, distributed=False, print_batch_step=50, save_res_path=f'{WORK}/pred.txt')
cfg['Optimizer']['lr'].update(learning_rate=LR, warmup_epoch=1)
cfg['Architecture']['Head']['head_list'][1]['NRTRHead']['max_text_length'] = 70
for sec in ('Train', 'Eval'):
    for t in cfg[sec]['dataset']['transforms']:
        for k, v in t.items():
            if isinstance(v, dict) and 'max_text_length' in v: v['max_text_length'] = 70
cfg['Train']['dataset'].update(data_dir=f'{WORK}/', label_file_list=[f'{WORK}/train_list.txt'])
cfg['Train']['sampler']['first_bs'] = BATCH
cfg['Train']['loader'].update(batch_size_per_card=BATCH, num_workers=NUM_WORKERS)
cfg['Eval']['dataset'].update(data_dir=f'{WORK}/', label_file_list=[f'{WORK}/eval_list.txt'])
cfg['Eval']['loader'].update(batch_size_per_card=16, num_workers=2)
yaml.safe_dump(cfg, open(f'{WORK}/rec_vi.yml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print('đã ghi', f'{WORK}/rec_vi.yml')


## 6. Train
Theo dõi `acc` (đúng nguyên dòng) và `norm_edit_dis` (càng cao càng tốt) trong log. Mô hình tốt nhất tự lưu là `best_accuracy`.

In [ ]:
# === 7. TRAIN (chạy lâu: ~4–6 phút/epoch trên T4 với 120k dòng). Mất kết nối thì chạy lại cell này, nó tự tiếp tục từ checkpoint. ===
resume = f'{OUT}/latest'
extra = f'-o Global.checkpoints={resume}' if os.path.exists(resume + '.pdparams') else ''
sh(f'python tools/train.py -c {WORK}/rec_vi.yml {extra}')


## 7. Xuất ONNX
Tạo `rec.onnx` (~8 MB, giống kích thước cũ) và `rec_dict.txt`.

In [ ]:
# === 8. Xuất ra ONNX (mô hình tốt nhất theo độ chính xác trên tập eval) ===
best = f'{OUT}/best_accuracy' if os.path.exists(f'{OUT}/best_accuracy.pdparams') else f'{OUT}/latest'
sh(f'python tools/export_model.py -c {WORK}/rec_vi.yml -o Global.pretrained_model={best} Global.save_inference_dir={WORK}/inf')
print(os.listdir(f'{WORK}/inf'))
model_file = 'inference.json' if os.path.exists(f'{WORK}/inf/inference.json') else 'inference.pdmodel'
sh(f'paddle2onnx --model_dir {WORK}/inf --model_filename {model_file} --params_filename inference.pdiparams '
   f'--save_file {WORK}/rec.onnx --opset_version 14')
open(f'{WORK}/rec_dict.txt', 'w', encoding='utf-8').write('\n'.join(DICT_VI) + '\n')       # app tự thêm dấu cách ở cuối
open(f'{WORK}/model.txt', 'w', encoding='utf-8').write('PP-OCRv5_mobile_det + latin_PP-OCRv5_mobile_rec (tinh chỉnh tiếng Việt: dataset niits/vietnamese-legal-ocr + câu chat đặt món)\n')
print('rec.onnx:', round(os.path.getsize(f'{WORK}/rec.onnx') / 1e6, 1), 'MB')


## 8. Chấm điểm
So sánh với mô hình gốc: CER có dấu ≈ 10.4%, CER bỏ dấu ≈ 10.4% (do chữ có dấu bị bỏ mất). Mục tiêu: CER có dấu < 3%. Nhóm `luat` có câu trùng với tập train (dataset chỉ có 1.301 câu) nên điểm cao hơn thực tế; nhóm `chat` và `tu_ngau_nhien` mới phản ánh đúng hơn.

In [ ]:
# === 8. Chấm mô hình ONNX vừa xuất trên tập eval (CER có dấu, CER bỏ dấu, đúng nguyên dòng) ===
import onnxruntime as ort, cv2, time
so = ort.SessionOptions(); so.log_severity_level = 3
sess = ort.InferenceSession(f'{WORK}/rec.onnx', so, providers=['CPUExecutionProvider'])
assert sess.get_outputs()[0].shape[-1] == len(DICT_VI) + 2, 'số lớp ONNX không khớp từ điển'
CH = ['<blank>'] + DICT_VI + [' ']
def infer(img):
    h, w = img.shape[:2]; nw = max(8, min(1600, int(math.ceil(48 * w / h))))
    x = cv2.resize(img, (nw, 48)).astype('float32')
    x = ((x / 255 - 0.5) / 0.5).transpose(2, 0, 1)[None]                       # BGR
    p = sess.run(None, {sess.get_inputs()[0].name: x})[0][0].argmax(-1)
    out, prev = [], 0
    for k in p:
        if k != prev and k != 0: out.append(CH[k])
        prev = k
    return ''.join(out)
def lev(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1): prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]
def strip(s): return ''.join(c for c in unicodedata.normalize('NFD', s.replace('đ', 'd').replace('Đ', 'D')) if unicodedata.category(c) != 'Mn')
rows = [l.rstrip('\n').split('\t', 1) for l in open(f'{WORK}/eval_list.txt', encoding='utf-8')]
cats = json.load(open(f'{WORK}/eval_cat.json'))
agg = collections.defaultdict(lambda: [0, 0, 0, 0, 0]); t0 = time.time(); sai = []
for (f, gt), cat in zip(rows, cats):
    pr = infer(cv2.imread(f'{WORK}/{f}'))
    if pr != gt and len(sai) < 12: sai.append((gt, pr))
    for key in (cat, 'TẤT CẢ'):
        a = agg[key]; a[0] += lev(pr, gt); a[1] += len(gt); a[2] += lev(strip(pr), strip(gt)); a[3] += pr == gt; a[4] += 1
print(f'{"nhóm":14s} {"dòng":>6s} {"CER có dấu":>11s} {"CER bỏ dấu":>11s} {"đúng cả dòng":>13s}')
for k, (e, n, e2, ok, m) in sorted(agg.items()):
    print(f'{k:14s} {m:6d} {100 * e / n:10.2f}% {100 * e2 / n:10.2f}% {100 * ok / m:12.1f}%')
print(f'tốc độ: {1000 * (time.time() - t0) / len(rows):.1f} ms/dòng (CPU của Colab)')
print('Mốc so sánh (mô hình gốc, ảnh chat mô phỏng): CER có dấu ≈ 10.4%, CER bỏ dấu ≈ 10.4%')
print('\nMột số dòng đọc sai (đáp án → máy đọc):')
for g, p in sai: print('  ', g, '→', p)


## 9. Đóng gói và đưa vào app
Cell dưới tải về `rec_vi_ket_qua.zip`. Giải nén rồi chép đè vào repo:

```
android/app/src/ocr/assets/ocr/rec.onnx
android/app/src/ocr/assets/ocr/rec_dict.txt
android/app/src/ocr/assets/ocr/model.txt
```
Giữ nguyên `det.onnx`. Build lại bản `ocr` rồi cài lên điện thoại ở chế độ TEST; mã Kotlin không cần sửa vì app đọc số lớp từ `rec_dict.txt`. Hoặc gửi file zip cho Claude để build và đưa lên nhánh `test`.

In [ ]:
# === 10. Đóng gói để đưa vào app ===
import shutil
pack = f'{WORK}/ket_qua'; os.makedirs(pack, exist_ok=True)
for f in ('rec.onnx', 'rec_dict.txt', 'model.txt'): shutil.copy(f'{WORK}/{f}', pack)
shutil.make_archive(f'{WORK}/rec_vi_ket_qua', 'zip', pack)
print('Đã đóng gói:', f'{WORK}/rec_vi_ket_qua.zip', os.listdir(pack))
try:
    from google.colab import files; files.download(f'{WORK}/rec_vi_ket_qua.zip')
except ImportError: pass
